# Colab Session B — Eval + Calibration + Figures

**Runtime:** L4 standard (or T4 — does not need A100)
**Est. time:** ~1.5h wall clock
**Tasks:**
1. Ablation study n=97
2. Recalibration n=97 → new Platt params
3. BioMistral re-eval n=97 (after local prompt/decoding fix)
4. Baseline re-eval with bootstrap CI
5. Regenerate all paper figures
6. Push results to Drive + GitHub

**Run in parallel with Session A** to save wall-clock time.  
Session B uses KB **v1** (already on Drive).


In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/kbssrikar7/final_project.git"
GITHUB_BRANCH  = "main"
DRIVE_BASE     = "/content/drive/MyDrive/healthcare_qa"
PROJECT_DIR    = "/content/project"
RESULTS_DIR    = f"{PROJECT_DIR}/evaluation/results"
FIGURES_DIR    = f"{PROJECT_DIR}/evaluation/figures"

import os
os.environ["HF_HUB_OFFLINE"]      = "1"   # avoid 40s retries
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("Configuration loaded")

In [ ]:
# ── B1: Environment Setup ─────────────────────────────────────────────────────
import subprocess, os

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clone / update repo
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", GITHUB_BRANCH], check=True)
os.chdir(PROJECT_DIR)
print("Repo ready")

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

In [ ]:
# ── B2: Mount Drive + Sync KB v1 + Models ────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import shutil, os

# KB v1
drive_kb = f"{DRIVE_BASE}/knowledge_base"
local_kb = f"{PROJECT_DIR}/data/knowledge_base"
if os.path.exists(drive_kb) and not os.path.exists(local_kb):
    print("Syncing KB v1 from Drive...")
    shutil.copytree(drive_kb, local_kb)
    print("KB v1 synced")

# Models (TinyLlama, BioMistral GGUF, LoRA adapters)
drive_models = f"{DRIVE_BASE}/models"
local_models = f"{PROJECT_DIR}/models"
if os.path.exists(drive_models):
    shutil.copytree(drive_models, local_models, dirs_exist_ok=True)
    print("Models synced from Drive")

# Disable HF offline now so HuggingFace model hub works if needed
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

In [ ]:
# ── B3: Ablation Study n=97 ──────────────────────────────────────────────────
# Expected time: ~25 min on L4
import subprocess, time

print("Running ablation study (n=97)...")
t0 = time.time()
result = subprocess.run([
    "python3", "evaluation/run_paper_eval.py",
    "--mode", "ablation",
    "--n", "97",
    "--model", "tinyllama",
], capture_output=False)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"Ablation complete in {elapsed/60:.1f} min")
    import json
    ab = json.loads(open(f"{RESULTS_DIR}/ablation.json").read())
    print(f"Ablation variants: {list(ab.keys())}")
else:
    print(f"WARNING: ablation returned rc={result.returncode}")

In [ ]:
# ── B4: Recalibration n=97 → new Platt params ────────────────────────────────
# Expected time: ~15 min
import subprocess, time

print("Running confidence calibration (n=97)...")
t0 = time.time()
result = subprocess.run([
    "python3", "evaluation/compute_calibration.py",
    "--n", "97",
], capture_output=False)
elapsed = time.time() - t0

if result.returncode == 0:
    import json
    cal = json.loads(open(f"{RESULTS_DIR}/calibration.json").read())
    print(f"Calibration complete in {elapsed/60:.1f} min")
    print(f"New Platt params: a={cal.get('platt_a'):.4f}, b={cal.get('platt_b'):.4f}")
    print(f"ECE raw={cal.get('ece_raw'):.4f} → calibrated={cal.get('ece_calibrated'):.4f}")
    print(f"n_samples={cal.get('n_samples')}")
else:
    print(f"WARNING: calibration returned rc={result.returncode}")

In [ ]:
# ── B5: BioMistral Re-eval n=97 ──────────────────────────────────────────────
# Expected time: ~25 min (BioMistral is slow even on GPU)
import subprocess, time
from pathlib import Path

gguf_path = Path(f"{PROJECT_DIR}/models/biomistral/ggml-model-Q4_K_M.gguf")
if not gguf_path.exists():
    print(f"SKIP: BioMistral GGUF not found at {gguf_path}")
else:
    print("Running BioMistral evaluation (n=97)...")
    t0 = time.time()
    result = subprocess.run([
        "python3", "evaluation/run_paper_eval.py",
        "--mode", "metrics",
        "--n", "97",
        "--model", "biomistral",
    ], capture_output=False)
    elapsed = time.time() - t0

    import json
    bio_path = f"{RESULTS_DIR}/metrics_full_biomistral.json"
    if result.returncode == 0 and Path(bio_path).exists():
        bio = json.loads(open(bio_path).read())
        kw = bio.get('keyword_coverage_mean', bio.get('keyword_coverage', 'N/A'))
        print(f"BioMistral re-eval complete in {elapsed/60:.1f} min")
        print(f"Keyword coverage: {kw:.4f}" if isinstance(kw, float) else f"Keyword coverage: {kw}")
    else:
        print(f"WARNING: BioMistral eval returned rc={result.returncode}")

In [ ]:
# ── B6: Baseline Re-eval with Bootstrap CI ───────────────────────────────────
# no-RAG / dense-only / no-XAI
# Expected time: ~20 min
import subprocess, time

print("Running baseline evaluations with bootstrap CI (n=97)...")
t0 = time.time()
result = subprocess.run([
    "python3", "evaluation/run_paper_eval.py",
    "--mode", "metrics",
    "--n", "97",
    "--variants", "standard",
    "--model", "tinyllama",
], capture_output=False)
elapsed = time.time() - t0

import json; import os
baseline_files = [f for f in os.listdir(RESULTS_DIR) if f.startswith("metrics_baseline")]
print(f"Baseline files present: {baseline_files}")
print(f"Evaluation complete in {elapsed/60:.1f} min")

In [ ]:
# ── B7: Regenerate Paper Figures ─────────────────────────────────────────────
import subprocess, time
from pathlib import Path

print("Regenerating paper figures...")
t0 = time.time()
result = subprocess.run(["python3", "evaluation/generate_paper_figures.py"], capture_output=False)
elapsed = time.time() - t0

figs = list(Path(FIGURES_DIR).glob("fig*.png")) if Path(FIGURES_DIR).exists() else []
print(f"Figures regenerated in {elapsed:.1f}s")
print(f"Figures present: {sorted(f.name for f in figs)}")

if len(figs) < 5:
    print(f"WARNING: Only {len(figs)} figures found, expected 6+")
else:
    print(f"OK: {len(figs)} figures ready")

In [ ]:
# ── B8: Sync Results to Drive + Push to GitHub ───────────────────────────────
import shutil, subprocess, os
from pathlib import Path

# Push results to Drive
drive_results = f"{DRIVE_BASE}/evaluation_results"
os.makedirs(drive_results, exist_ok=True)
for f in Path(RESULTS_DIR).glob("*.json"):
    shutil.copy2(f, drive_results)
for f in Path(FIGURES_DIR).glob("*.png"):
    shutil.copy2(f, f"{DRIVE_BASE}/figures/{f.name}") if Path(f"{DRIVE_BASE}/figures").mkdir(parents=True, exist_ok=True) or True else None
print(f"Results synced to Drive: {drive_results}")

# Commit + push to GitHub
try:
    subprocess.run(["git", "-C", PROJECT_DIR, "config", "user.email", "colab@healthcare-qa"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "config", "user.name", "Colab Session B"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "add",
                    "evaluation/results/", "evaluation/figures/"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "commit", "-m",
                    "feat: Colab Session B — ablation n=97, recalibration n=97, BioMistral re-eval, figures"],
                   check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "push", "origin", GITHUB_BRANCH], check=True)
    print("Results pushed to GitHub")
except subprocess.CalledProcessError as e:
    print(f"WARNING: Git push failed: {e}. Results are safe on Drive.")

print("Session B complete. When Session A also finishes, run Session C notebook.")